# DeepPoly MNIST Robustness Bounds

This notebook keeps the experiment focused on certification.

The proof condition for an image with true label `y` is:

`lower[y] > upper[j]` for every `j != y`.

The notebook compares a normally trained model against a model trained with a DeepPoly bound loss. FGSM is included only as an empirical sanity check; DeepPoly bounds are the certification result.

In [ ]:
import gzip
import os
import random
import struct
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("mnist")
BASELINE_MODEL_PATH = Path("mnist_baseline.pt")
ROBUST_MODEL_PATH = Path("mnist_deeppoly_robust.pt")

RESET_MODELS = True  # set False after one full run to reuse saved weights

IMAGE_INDEX = 0
EPSILON = 0.05
EVAL_EPSILONS = [0.00, 0.01, 0.03, 0.05]

TRAIN_LIMIT = 12000
ROBUST_TRAIN_LIMIT = 12000
CERT_LIMIT = 500
FGSM_LIMIT = 500

BASELINE_EPOCHS = 5
ROBUST_EPOCHS = 30
BASELINE_BATCH = 128
ROBUST_BATCH = 64
BASELINE_LR = 1e-3
ROBUST_LR = 5e-4
ROBUST_LAMBDA = 0.5

print(f"device={DEVICE}")

In [ ]:
BASE_URL = "https://ossci-datasets.s3.amazonaws.com/mnist"
MNIST_FILES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]


def ensure_mnist(data_dir=DATA_DIR):
    data_dir.mkdir(exist_ok=True)
    for filename in MNIST_FILES:
        raw_path = data_dir / filename[:-3]
        gz_path = data_dir / filename
        if raw_path.exists():
            continue

        print(f"downloading {filename}")
        urllib.request.urlretrieve(f"{BASE_URL}/{filename}", gz_path)
        with gzip.open(gz_path, "rb") as src, open(raw_path, "wb") as dst:
            dst.write(src.read())
        gz_path.unlink()


def read_images(path):
    with open(path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"bad image file: {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows * cols).astype(np.float32) / 255.0


def read_labels(path):
    with open(path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"bad label file: {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int64)


def load_mnist(data_dir=DATA_DIR):
    ensure_mnist(data_dir)
    x_train = read_images(data_dir / "train-images-idx3-ubyte")
    y_train = read_labels(data_dir / "train-labels-idx1-ubyte")
    x_test = read_images(data_dir / "t10k-images-idx3-ubyte")
    y_test = read_labels(data_dir / "t10k-labels-idx1-ubyte")

    x_train = torch.from_numpy(x_train).to(DEVICE)
    y_train = torch.from_numpy(y_train).to(DEVICE)
    x_test = torch.from_numpy(x_test).to(DEVICE)
    y_test = torch.from_numpy(y_test).to(DEVICE)
    return x_train, y_train, x_test, y_test


X_train, y_train, X_test, y_test = load_mnist()
print(f"train={tuple(X_train.shape)}  test={tuple(X_test.shape)}")

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 10),
        )

    def forward(self, x):
        return self.layers(x)


def batched_indices(n, batch_size, shuffle=True):
    if shuffle:
        order = torch.randperm(n, device=DEVICE)
    else:
        order = torch.arange(n, device=DEVICE)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


@torch.no_grad()
def clean_accuracy(model, x, y, limit=None, batch_size=512):
    model.eval()
    if limit is not None:
        x = x[:limit]
        y = y[:limit]
    correct = 0
    total = 0
    for idx in batched_indices(len(x), batch_size, shuffle=False):
        pred = model(x[idx]).argmax(dim=1)
        correct += (pred == y[idx]).sum().item()
        total += len(idx)
    return correct / total


def train_clean(model, x, y, epochs, batch_size, lr, limit):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x = x[:limit]
    y = y[:limit]

    for epoch in range(1, epochs + 1):
        losses = []
        model.train()
        for idx in batched_indices(len(x), batch_size, shuffle=True):
            logits = model(x[idx])
            loss = F.cross_entropy(logits, y[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        acc = clean_accuracy(model, X_test, y_test, limit=1000)
        print(f"clean epoch {epoch:02d}/{epochs}  loss={np.mean(losses):.4f}  test_acc_1k={100 * acc:.2f}%")


def save_model(model, path):
    torch.save(model.state_dict(), path)


def load_model(path):
    model = MLP().to(DEVICE)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    return model

In [ ]:
def eval_symbolic(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u):
    lower = lb_bias + torch.where(lb_coef >= 0, lb_coef * inp_l[:, None, :], lb_coef * inp_u[:, None, :]).sum(dim=2)
    upper = ub_bias + torch.where(ub_coef >= 0, ub_coef * inp_u[:, None, :], ub_coef * inp_l[:, None, :]).sum(dim=2)
    return lower, upper


def dp_first_linear(layer, inp_l, inp_u):
    W = layer.weight
    b = layer.bias
    batch = inp_l.shape[0]
    lb_coef = W.unsqueeze(0).expand(batch, -1, -1)
    ub_coef = W.unsqueeze(0).expand(batch, -1, -1)
    lb_bias = b.unsqueeze(0).expand(batch, -1)
    ub_bias = b.unsqueeze(0).expand(batch, -1)
    lower, upper = eval_symbolic(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u)
    return lb_coef, ub_coef, lb_bias, ub_bias, lower, upper


def dp_linear(prev, layer, inp_l, inp_u):
    prev_lbc, prev_ubc, prev_lbb, prev_ubb, _, _ = prev
    W = layer.weight
    b = layer.bias
    W_pos = torch.clamp(W, min=0.0)
    W_neg = torch.clamp(W, max=0.0)

    lb_coef = torch.einsum("on,bnd->bod", W_pos, prev_lbc) + torch.einsum("on,bnd->bod", W_neg, prev_ubc)
    ub_coef = torch.einsum("on,bnd->bod", W_pos, prev_ubc) + torch.einsum("on,bnd->bod", W_neg, prev_lbc)
    lb_bias = torch.einsum("on,bn->bo", W_pos, prev_lbb) + torch.einsum("on,bn->bo", W_neg, prev_ubb) + b
    ub_bias = torch.einsum("on,bn->bo", W_pos, prev_ubb) + torch.einsum("on,bn->bo", W_neg, prev_lbb) + b

    lower, upper = eval_symbolic(lb_coef, ub_coef, lb_bias, ub_bias, inp_l, inp_u)
    return lb_coef, ub_coef, lb_bias, ub_bias, lower, upper


def dp_relu(prev):
    prev_lbc, prev_ubc, prev_lbb, prev_ubb, lower, upper = prev
    zeros_coef = torch.zeros_like(prev_lbc)
    zeros_bias = torch.zeros_like(prev_lbb)

    positive = lower >= 0
    negative = upper <= 0
    mixed = ~(positive | negative)

    use_identity_lower = positive | (mixed & (upper > -lower))
    denom = torch.clamp(upper - lower, min=1e-12)
    slope = torch.where(mixed, upper / denom, torch.ones_like(upper))
    intercept = torch.where(mixed, -upper * lower / denom, torch.zeros_like(upper))

    lb_coef = torch.where(use_identity_lower[:, :, None], prev_lbc, zeros_coef)
    lb_bias = torch.where(use_identity_lower, prev_lbb, zeros_bias)

    mixed_ubc = slope[:, :, None] * prev_ubc
    mixed_ubb = slope * prev_ubb + intercept
    ub_coef = torch.where(positive[:, :, None], prev_ubc, torch.where(mixed[:, :, None], mixed_ubc, zeros_coef))
    ub_bias = torch.where(positive, prev_ubb, torch.where(mixed, mixed_ubb, zeros_bias))

    relu_lower = torch.where(use_identity_lower, lower, torch.zeros_like(lower))
    relu_upper = torch.where(positive | mixed, upper, torch.zeros_like(upper))
    return lb_coef, ub_coef, lb_bias, ub_bias, relu_lower, relu_upper


def deeppoly_bounds(model, x, epsilon):
    inp_l = torch.clamp(x - epsilon, 0.0, 1.0)
    inp_u = torch.clamp(x + epsilon, 0.0, 1.0)

    layers = list(model.layers)
    curr = None
    for layer in layers:
        if isinstance(layer, nn.Linear):
            curr = dp_first_linear(layer, inp_l, inp_u) if curr is None else dp_linear(curr, layer, inp_l, inp_u)
        elif isinstance(layer, nn.ReLU):
            curr = dp_relu(curr)
        else:
            raise TypeError(f"unsupported layer: {layer}")

    return curr[4], curr[5]


def certified_margin(lower, upper, y):
    true_lower = lower.gather(1, y[:, None]).squeeze(1)
    wrong_mask = F.one_hot(y, num_classes=10).bool()
    max_wrong_upper = upper.masked_fill(wrong_mask, -1e9).max(dim=1).values
    return true_lower - max_wrong_upper


def deeppoly_robust_loss(model, x, y, epsilon):
    lower, upper = deeppoly_bounds(model, x, epsilon)
    true_lower = lower.gather(1, y[:, None]).squeeze(1)
    wrong_mask = F.one_hot(y, num_classes=10).bool()
    smooth_wrong_upper = torch.logsumexp(upper.masked_fill(wrong_mask, -1e9), dim=1)
    return F.softplus(smooth_wrong_upper - true_lower).mean()


@torch.no_grad()
def bounds_table(model, x, y, epsilon):
    model.eval()
    lower, upper = deeppoly_bounds(model, x[None, :], epsilon)
    lower = lower[0].detach().cpu().numpy()
    upper = upper[0].detach().cpu().numpy()
    df = pd.DataFrame({
        "digit": np.arange(10),
        "lower_bound": lower,
        "upper_bound": upper,
        "true_digit": np.arange(10) == int(y),
    })
    return df


@torch.no_grad()
def print_bound_summary(model, x, y, epsilon):
    df = bounds_table(model, x, y, epsilon)
    true_label = int(y)
    true_lower = float(df.loc[df.digit == true_label, "lower_bound"].iloc[0])
    max_wrong_upper = float(df.loc[df.digit != true_label, "upper_bound"].max())
    margin = true_lower - max_wrong_upper
    pred = int(model(x[None, :]).argmax(dim=1).item())
    print(f"image_index={IMAGE_INDEX}  true_label={true_label}  clean_pred={pred}  epsilon={epsilon}")
    print(f"true_lower={true_lower:.6f}  max_wrong_upper={max_wrong_upper:.6f}  margin={margin:.6f}  certified={margin > 0}")
    display(df.style.format({"lower_bound": "{:.6f}", "upper_bound": "{:.6f}"}))


@torch.no_grad()
def certification_report(model, x, y, epsilons, limit=CERT_LIMIT, batch_size=32):
    model.eval()
    x = x[:limit]
    y = y[:limit]
    rows = []

    for epsilon in epsilons:
        clean_correct = 0
        certified = 0
        margins = []

        for idx in batched_indices(len(x), batch_size, shuffle=False):
            xb = x[idx]
            yb = y[idx]
            pred = model(xb).argmax(dim=1)
            lower, upper = deeppoly_bounds(model, xb, epsilon)
            margin = certified_margin(lower, upper, yb)

            clean_correct += (pred == yb).sum().item()
            certified += (margin > 0).sum().item()
            margins.append(margin.detach().cpu())

        margins = torch.cat(margins)
        rows.append({
            "epsilon": epsilon,
            "clean_acc": clean_correct / len(x),
            "certified_acc": certified / len(x),
            "avg_margin": float(margins.mean()),
            "min_margin": float(margins.min()),
        })

    return pd.DataFrame(rows)

## 1. Baseline Model

Train normally with cross entropy. Then run DeepPoly and print the final digit bounds.

In [ ]:
if BASELINE_MODEL_PATH.exists() and not RESET_MODELS:
    baseline_model = load_model(BASELINE_MODEL_PATH)
else:
    baseline_model = MLP().to(DEVICE)
    train_clean(baseline_model, X_train, y_train, BASELINE_EPOCHS, BASELINE_BATCH, BASELINE_LR, TRAIN_LIMIT)
    save_model(baseline_model, BASELINE_MODEL_PATH)

print(f"baseline_test_acc={100 * clean_accuracy(baseline_model, X_test, y_test):.2f}%")
print_bound_summary(baseline_model, X_test[IMAGE_INDEX], y_test[IMAGE_INDEX], EPSILON)
display(certification_report(baseline_model, X_test, y_test, EVAL_EPSILONS).style.format({
    "clean_acc": "{:.2%}",
    "certified_acc": "{:.2%}",
    "avg_margin": "{:.4f}",
    "min_margin": "{:.4f}",
}))

## 2. DeepPoly Robust Training

The robust objective directly penalizes failed certificates:

`robust_loss = softplus(logsumexp(upper_wrong_digits) - lower_true_digit)`

The model is trained with:

`clean_cross_entropy + lambda * robust_loss`

Epsilon is increased gradually during robust training so the model does not see the hardest bound problem on the first epoch.

In [ ]:
def train_deeppoly_robust(model, x, y, epochs, batch_size, lr, epsilon, lam, limit):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    x = x[:limit]
    y = y[:limit]

    for epoch in range(1, epochs + 1):
        current_eps = epsilon * epoch / epochs
        clean_losses = []
        robust_losses = []

        model.train()
        for idx in batched_indices(len(x), batch_size, shuffle=True):
            xb = x[idx]
            yb = y[idx]
            logits = model(xb)
            clean_loss = F.cross_entropy(logits, yb)
            robust_loss = deeppoly_robust_loss(model, xb, yb, current_eps)
            loss = clean_loss + lam * robust_loss

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            clean_losses.append(clean_loss.item())
            robust_losses.append(robust_loss.item())

        acc = clean_accuracy(model, X_test, y_test, limit=1000)
        print(
            f"robust epoch {epoch:02d}/{epochs}  eps={current_eps:.4f}  "
            f"clean_loss={np.mean(clean_losses):.4f}  "
            f"robust_loss={np.mean(robust_losses):.4f}  "
            f"test_acc_1k={100 * acc:.2f}%"
        )


if ROBUST_MODEL_PATH.exists() and not RESET_MODELS:
    robust_model = load_model(ROBUST_MODEL_PATH)
else:
    robust_model = MLP().to(DEVICE)
    train_deeppoly_robust(
        robust_model,
        X_train,
        y_train,
        ROBUST_EPOCHS,
        ROBUST_BATCH,
        ROBUST_LR,
        EPSILON,
        ROBUST_LAMBDA,
        ROBUST_TRAIN_LIMIT,
    )
    save_model(robust_model, ROBUST_MODEL_PATH)

print(f"robust_test_acc={100 * clean_accuracy(robust_model, X_test, y_test):.2f}%")
print_bound_summary(robust_model, X_test[IMAGE_INDEX], y_test[IMAGE_INDEX], EPSILON)
display(certification_report(robust_model, X_test, y_test, EVAL_EPSILONS).style.format({
    "clean_acc": "{:.2%}",
    "certified_acc": "{:.2%}",
    "avg_margin": "{:.4f}",
    "min_margin": "{:.4f}",
}))

## 3. FGSM Sanity Check

FGSM is not a certificate. It is included only to show whether a simple empirical attack succeeds before and after DeepPoly robust training.

In [ ]:
def fgsm_attack(model, x, y, epsilon):
    x_adv = x.detach().clone().requires_grad_(True)
    loss = F.cross_entropy(model(x_adv), y)
    model.zero_grad(set_to_none=True)
    loss.backward()
    return torch.clamp(x_adv + epsilon * x_adv.grad.sign(), 0.0, 1.0).detach()


@torch.no_grad()
def _predict(model, x):
    return model(x).argmax(dim=1)


def fgsm_report(model, x, y, epsilons, limit=FGSM_LIMIT, batch_size=128):
    model.eval()
    x = x[:limit]
    y = y[:limit]
    rows = []

    for epsilon in epsilons:
        clean_correct = 0
        adv_correct = 0
        successful_attacks = 0
        clean_correct_total = 0

        for idx in batched_indices(len(x), batch_size, shuffle=False):
            xb = x[idx]
            yb = y[idx]
            pred_clean = _predict(model, xb)
            x_adv = fgsm_attack(model, xb, yb, epsilon)
            pred_adv = _predict(model, x_adv)

            clean_ok = pred_clean == yb
            adv_ok = pred_adv == yb
            clean_correct += clean_ok.sum().item()
            adv_correct += adv_ok.sum().item()
            clean_correct_total += clean_ok.sum().item()
            successful_attacks += (clean_ok & ~adv_ok).sum().item()

        rows.append({
            "epsilon": epsilon,
            "clean_acc": clean_correct / len(x),
            "fgsm_acc": adv_correct / len(x),
            "attack_success_on_clean_correct": successful_attacks / max(1, clean_correct_total),
        })

    return pd.DataFrame(rows)


print("baseline FGSM")
display(fgsm_report(baseline_model, X_test, y_test, EVAL_EPSILONS).style.format({
    "clean_acc": "{:.2%}",
    "fgsm_acc": "{:.2%}",
    "attack_success_on_clean_correct": "{:.2%}",
}))

print("DeepPoly-trained FGSM")
display(fgsm_report(robust_model, X_test, y_test, EVAL_EPSILONS).style.format({
    "clean_acc": "{:.2%}",
    "fgsm_acc": "{:.2%}",
    "attack_success_on_clean_correct": "{:.2%}",
}))

## 4. Final Bound Comparison

This is the direct comparison for the selected image. Each digit has its baseline DeepPoly output range and its post-training DeepPoly output range.

In [ ]:
@torch.no_grad()
def final_bound_comparison(baseline_model, robust_model, x, y, epsilon):
    y_int = int(y.detach().cpu().item())
    baseline = bounds_table(baseline_model, x, y, epsilon).rename(columns={
        "lower_bound": "baseline_lower",
        "upper_bound": "baseline_upper",
    })
    trained = bounds_table(robust_model, x, y, epsilon).rename(columns={
        "lower_bound": "trained_lower",
        "upper_bound": "trained_upper",
    })

    comparison = baseline[["digit", "baseline_lower", "baseline_upper"]].merge(
        trained[["digit", "trained_lower", "trained_upper"]],
        on="digit",
    )
    comparison["lower_change"] = comparison["trained_lower"] - comparison["baseline_lower"]
    comparison["upper_change"] = comparison["trained_upper"] - comparison["baseline_upper"]
    comparison["true_digit"] = comparison["digit"] == y_int
    return comparison


comparison = final_bound_comparison(
    baseline_model,
    robust_model,
    X_test[IMAGE_INDEX],
    y_test[IMAGE_INDEX],
    EPSILON,
)

print(f"image_index={IMAGE_INDEX}  true_label={int(y_test[IMAGE_INDEX].detach().cpu().item())}  epsilon={EPSILON}")
display(comparison.style.format({
    "baseline_lower": "{:.6f}",
    "baseline_upper": "{:.6f}",
    "trained_lower": "{:.6f}",
    "trained_upper": "{:.6f}",
    "lower_change": "{:.6f}",
    "upper_change": "{:.6f}",
}))

## 5. Final Aggregate Summary

This is the main result table: baseline versus DeepPoly-trained model at the target epsilon.

In [ ]:
def final_aggregate_summary(baseline_model, robust_model, epsilon):
    rows = []
    for name, model in [("baseline", baseline_model), ("deeppoly_trained", robust_model)]:
        cert = certification_report(model, X_test, y_test, [epsilon], limit=CERT_LIMIT).iloc[0]
        fgsm = fgsm_report(model, X_test, y_test, [epsilon], limit=FGSM_LIMIT).iloc[0]
        rows.append({
            "model": name,
            "epsilon": epsilon,
            "clean_acc": cert["clean_acc"],
            "certified_acc": cert["certified_acc"],
            "avg_margin": cert["avg_margin"],
            "min_margin": cert["min_margin"],
            "fgsm_acc": fgsm["fgsm_acc"],
            "fgsm_attack_success": fgsm["attack_success_on_clean_correct"],
        })
    return pd.DataFrame(rows)


summary = final_aggregate_summary(baseline_model, robust_model, EPSILON)
display(summary.style.format({
    "epsilon": "{:.3f}",
    "clean_acc": "{:.2%}",
    "certified_acc": "{:.2%}",
    "avg_margin": "{:.4f}",
    "min_margin": "{:.4f}",
    "fgsm_acc": "{:.2%}",
    "fgsm_attack_success": "{:.2%}",
}))

## 6. Visual Summary

These figures show the same results as the tables: the selected input region, the output intervals for each digit, certification behavior across epsilon, and the margin distribution before and after DeepPoly training.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "axes.titleweight": "semibold",
    "font.size": 10,
})

COLOR_BASELINE = "#64748b"
COLOR_TRAINED = "#2563eb"
COLOR_TRUE = "#16a34a"
COLOR_FAIL = "#dc2626"
COLOR_ZERO = "#111827"


def margin_values(model, x, y, epsilon, limit=CERT_LIMIT, batch_size=32):
    model.eval()
    x = x[:limit]
    y = y[:limit]
    vals = []
    with torch.no_grad():
        for idx in batched_indices(len(x), batch_size, shuffle=False):
            lower, upper = deeppoly_bounds(model, x[idx], epsilon)
            vals.append(certified_margin(lower, upper, y[idx]).detach().cpu())
    return torch.cat(vals).numpy()


def style_axis(ax, title=None, xlabel=None, ylabel=None):
    if title:
        ax.set_title(title, loc="left", pad=10)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.tick_params(length=0)
    return ax

In [ ]:
sample = X_test[IMAGE_INDEX].detach().cpu().view(28, 28).numpy()
lower_img = np.clip(sample - EPSILON, 0.0, 1.0)
upper_img = np.clip(sample + EPSILON, 0.0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(8.4, 2.8), constrained_layout=True)
for ax, img, title in zip(
    axes,
    [sample, lower_img, upper_img],
    ["Original image", "Input lower bound", "Input upper bound"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    f"Input region for image {IMAGE_INDEX} | true label {int(y_test[IMAGE_INDEX].detach().cpu().item())} | epsilon={EPSILON}",
    fontsize=11,
    fontweight="semibold",
)
plt.show()

In [ ]:
def plot_bound_intervals(comparison, true_label):
    fig, ax = plt.subplots(figsize=(9.4, 5.0))
    digits = comparison["digit"].to_numpy()

    for _, row in comparison.iterrows():
        digit = int(row["digit"])
        is_true = digit == true_label
        y_base = digit + 0.17
        y_train = digit - 0.17
        base_color = COLOR_TRUE if is_true else COLOR_BASELINE
        train_color = COLOR_TRUE if is_true else COLOR_TRAINED

        ax.hlines(y_base, row["baseline_lower"], row["baseline_upper"], color=base_color, lw=3.0, alpha=0.65)
        ax.scatter([row["baseline_lower"], row["baseline_upper"]], [y_base, y_base], color=base_color, s=16, alpha=0.9)
        ax.hlines(y_train, row["trained_lower"], row["trained_upper"], color=train_color, lw=3.0, alpha=0.95)
        ax.scatter([row["trained_lower"], row["trained_upper"]], [y_train, y_train], color=train_color, s=18, alpha=0.95)

    true_row = comparison.loc[comparison["digit"] == true_label].iloc[0]
    max_wrong_upper = comparison.loc[comparison["digit"] != true_label, "trained_upper"].max()
    ax.axvline(true_row["trained_lower"], color=COLOR_TRUE, lw=1.4, ls="--", label="trained true lower")
    ax.axvline(max_wrong_upper, color=COLOR_FAIL, lw=1.4, ls="--", label="trained max wrong upper")

    ax.set_yticks(digits)
    ax.set_yticklabels([str(d) for d in digits])
    ax.invert_yaxis()
    ax.legend(frameon=False, loc="lower right")
    style_axis(ax, "DeepPoly output intervals by digit", "logit bound", "digit")
    plt.show()


plot_bound_intervals(comparison, int(y_test[IMAGE_INDEX].detach().cpu().item()))

In [ ]:
baseline_eps_report = certification_report(baseline_model, X_test, y_test, EVAL_EPSILONS)
trained_eps_report = certification_report(robust_model, X_test, y_test, EVAL_EPSILONS)

fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.plot(
    baseline_eps_report["epsilon"],
    baseline_eps_report["certified_acc"],
    marker="o",
    lw=2.2,
    color=COLOR_BASELINE,
    label="baseline certified",
)
ax.plot(
    trained_eps_report["epsilon"],
    trained_eps_report["certified_acc"],
    marker="o",
    lw=2.2,
    color=COLOR_TRAINED,
    label="DeepPoly-trained certified",
)
ax.plot(
    baseline_eps_report["epsilon"],
    baseline_eps_report["clean_acc"],
    marker=".",
    lw=1.6,
    ls="--",
    color=COLOR_BASELINE,
    alpha=0.55,
    label="baseline clean",
)
ax.plot(
    trained_eps_report["epsilon"],
    trained_eps_report["clean_acc"],
    marker=".",
    lw=1.6,
    ls="--",
    color=COLOR_TRAINED,
    alpha=0.55,
    label="DeepPoly-trained clean",
)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, ncol=2, loc="lower left")
style_axis(ax, "Certified accuracy across epsilon", "epsilon", "accuracy")
plt.show()

In [ ]:
metric_cols = ["clean_acc", "certified_acc", "fgsm_acc"]
metric_labels = ["Clean", "Certified", "FGSM"]
x_pos = np.arange(len(metric_cols))
width = 0.36

baseline_vals = summary.loc[summary["model"] == "baseline", metric_cols].iloc[0].to_numpy(dtype=float)
trained_vals = summary.loc[summary["model"] == "deeppoly_trained", metric_cols].iloc[0].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.4, 4.2))
ax.bar(x_pos - width / 2, baseline_vals, width, color=COLOR_BASELINE, label="baseline")
ax.bar(x_pos + width / 2, trained_vals, width, color=COLOR_TRAINED, label="DeepPoly-trained")
ax.set_xticks(x_pos)
ax.set_xticklabels(metric_labels)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, loc="upper left")
style_axis(ax, f"Final metrics at epsilon={EPSILON}", None, "accuracy")
plt.show()

In [ ]:
baseline_margins = margin_values(baseline_model, X_test, y_test, EPSILON)
trained_margins = margin_values(robust_model, X_test, y_test, EPSILON)

fig, ax = plt.subplots(figsize=(8.2, 4.2))
bins = np.linspace(min(baseline_margins.min(), trained_margins.min()), max(baseline_margins.max(), trained_margins.max()), 24)
ax.hist(baseline_margins, bins=bins, color=COLOR_BASELINE, alpha=0.42, label="baseline")
ax.hist(trained_margins, bins=bins, color=COLOR_TRAINED, alpha=0.42, label="DeepPoly-trained")
ax.axvline(0.0, color=COLOR_ZERO, lw=1.3, ls="--", label="certification threshold")
ax.legend(frameon=False)
style_axis(ax, f"Certification margin distribution at epsilon={EPSILON}", "margin", "count")
plt.show()